In [ ]:
# =========================================================
# INSTALLS (run once in colab)
# =========================================================

!pip -q install bert-score rouge-score sentence-transformers nltk

# =========================================================
# IMPORTS
# =========================================================

import json
import ast
import re
import numpy as np

from collections import defaultdict

from nltk.translate.bleu_score import sentence_bleu,SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# FILE PATHS
# =========================================================

GT_PATH="/content/Validation_Stories.txt"
PRED_PATH="/content/validation_predictions (1).json"


# =========================================================
# LOAD DATA
# =========================================================

def load_gt(path):
    with open(path,"r",encoding="utf-8") as f:
        txt=f.read()

    try:
        return json.loads(txt)
    except:
        return ast.literal_eval(txt)


def load_pred(path):
    with open(path,"r",encoding="utf-8") as f:
        return json.load(f)


gt_data=load_gt(GT_PATH)
pred_data=load_pred(PRED_PATH)

print("GT:",len(gt_data))
print("Pred:",len(pred_data))


# =========================================================
# EXTRACT JSON FROM PREDICTIONS
# =========================================================

def extract_json(text):

    if text is None:
        return None

    text=str(text)

    text=text.replace("```json","").replace("```","")

    try:
        return json.loads(text)
    except:
        pass

    candidates=re.findall(r"\[.*\]|\{.*\}",text,re.DOTALL)

    for c in reversed(candidates):
        try:
            return json.loads(c)
        except:
            continue

    return None


# =========================================================
# NORMALIZE SCENES
# =========================================================

def normalize(data):

    if isinstance(data,dict):

        if "output" in data:
            data=data["output"]

        elif "scenes" in data:
            data=data["scenes"]

    if isinstance(data,dict):
        data=[data]

    if not isinstance(data,list):
        return []

    scenes=[]

    for s in data:

        if not isinstance(s,dict):
            continue

        scenes.append({

            "emotion":str(
                s.get("emotion","")
            ).lower().strip(),

            "setting":str(
                s.get("setting","")
            ).lower().strip(),

            "atmosphere":str(
                s.get("atmosphere","")
            ).lower().strip(),

            "state":str(
                s.get("state","")
            ).lower().strip(),

            "theme":str(
                s.get("theme","")
            ).lower().strip(),

            "state_change":str(
                s.get("state_change","")
            ).lower().strip()
        })

    return scenes


# =========================================================
# PRF1
# =========================================================

def prf(gt,pred):

    gt=set([x for x in gt if x!=""])
    pred=set([x for x in pred if x!=""])

    tp=len(gt & pred)
    fp=len(pred-gt)
    fn=len(gt-pred)

    p=tp/(tp+fp+1e-9)
    r=tp/(tp+fn+1e-9)
    f1=2*p*r/(p+r+1e-9)

    return p,r,f1


# =========================================================
# AUXILIARY METRICS
# =========================================================

def scene_acc(gt,pred):
    return 1-abs(len(gt)-len(pred))/max(len(gt),1)


def state_transition(gt,pred):

    g=[x["state_change"] for x in gt if x["state_change"]]
    p=[x["state_change"] for x in pred if x["state_change"]]

    return prf(g,p)[2]


def causal(gt,pred):

    gs=[x["state"] for x in gt]
    ps=[x["state"] for x in pred]

    n=min(len(gs),len(ps))

    if n==0:
        return 0

    return sum(gs[i]==ps[i] for i in range(n))/n


def schema_valid(pred):

    req=["emotion","setting","atmosphere","state","theme"]

    total=0
    valid=0

    for s in pred:
        for k in req:
            total+=1

            if s[k]!="":
                valid+=1

    return valid/(total+1e-9)


# =========================================================
# MAIN STRUCTURAL EVALUATION
# =========================================================

results=defaultdict(list)

valid_json=0
exact_match=0
struct_match=0
scene_counts=[]

refs=[]
hyps=[]

for gt_story,pred_story in zip(gt_data,pred_data):

    gt=normalize(gt_story)

    pred_json=extract_json(
        pred_story["predicted_output"]
    )

    if pred_json is None:
        continue

    valid_json+=1

    pred=normalize(pred_json)

    scene_counts.append(len(pred))

    if len(gt)==len(pred):
        exact_match+=1

    if len(gt)==len(pred):
        struct_match+=1

    results["scene_acc"].append(
        scene_acc(gt,pred)
    )

    for field in [
        "emotion",
        "setting",
        "atmosphere",
        "state",
        "theme"
    ]:

        p,r,f1=prf(
            [x[field] for x in gt],
            [x[field] for x in pred]
        )

        results[field+"_precision"].append(p)
        results[field+"_recall"].append(r)
        results[field+"_f1"].append(f1)

    results["state_transition"].append(
        state_transition(gt,pred)
    )

    results["causal"].append(
        causal(gt,pred)
    )

    results["schema"].append(
        schema_valid(pred)
    )

    refs.append(json.dumps(gt))
    hyps.append(json.dumps(pred))


# =========================================================
# ROUGE
# =========================================================

rouger=rouge_scorer.RougeScorer(
["rouge1","rougeL"],
use_stemmer=True
)

r1=[]
rl=[]

for ref,hyp in zip(refs,hyps):

    s=rouger.score(ref,hyp)

    r1.append(
        s["rouge1"].fmeasure
    )

    rl.append(
        s["rougeL"].fmeasure
    )


# =========================================================
# BLEU
# =========================================================

smooth=SmoothingFunction().method1

bleus=[]

for ref,hyp in zip(refs,hyps):

    bleus.append(
        sentence_bleu(
            [ref.split()],
            hyp.split(),
            smoothing_function=smooth
        )
    )


# =========================================================
# BERTSCORE
# =========================================================

P,R,F=bertscore(
hyps,
refs,
lang="en",
verbose=False
)

berts=np.mean(F.numpy())


# =========================================================
# SENTENCE SIMILARITY
# =========================================================

model=SentenceTransformer(
"all-MiniLM-L6-v2"
)

emb1=model.encode(refs)
emb2=model.encode(hyps)

sims=[]

for a,b in zip(emb1,emb2):

    sims.append(
        cosine_similarity(
            [a],[b]
        )[0][0]
    )


# =========================================================
# PRINT RESULTS
# =========================================================

print("\n=========== STRUCTURAL METRICS ===========\n")

for k,v in sorted(results.items()):
    print(
        f"{k}: {np.mean(v):.4f}"
    )


print("\n=========== VALIDATION QUALITY METRICS ===========\n")

print(
"Exact Match:",
round(
exact_match/len(gt_data),4
)
)

print(
"Structural Match:",
round(
struct_match/len(gt_data),4
)
)

print(
"Valid JSON %:",
round(
valid_json/len(gt_data),4
)
)

print(
"Avg Scenes/Story:",
round(
np.mean(scene_counts),4
)
)

print(
"ROUGE-1:",
round(np.mean(r1),4)
)

print(
"ROUGE-L:",
round(np.mean(rl),4)
)

print(
"BLEU:",
round(np.mean(bleus),4)
)

print(
"BERTScore:",
round(berts,4)
)

print(
"Sentence Similarity:",
round(np.mean(sims),4)
)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
GT: 40
Pred: 40


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=========== STRUCTURAL METRICS ===========

atmosphere_f1: 0.9198
atmosphere_precision: 0.8904
atmosphere_recall: 0.9649
causal: 0.4956
emotion_f1: 0.8070
emotion_precision: 0.8053
emotion_recall: 0.8487
scene_acc: 0.7961
schema: 1.0000
setting_f1: 0.8070
setting_precision: 0.8158
setting_recall: 0.8026
state_f1: 0.9189
state_precision: 0.8890
state_recall: 0.9671
state_transition: 0.6361
theme_f1: 0.5000
theme_precision: 0.5000
theme_recall: 0.5000

=========== VALIDATION QUALITY METRICS ===========

Exact Match: 0.45
Structural Match: 0.45
Valid JSON %: 0.95
Avg Scenes/Story: 3.8421
ROUGE-1: 0.8407
ROUGE-L: 0.8015
BLEU: 0.6396
BERTScore: 0.9707
Sentence Similarity: 0.983
